# 61 — Baseline vectorielle par passages

Mesure la récupération du pipeline A lorsque l'unité d'indexation est un **passage** de quelques phrases, titre du document en tête, au lieu du document entier.

Le modèle d'encodage tronque ses entrées à 350 tokens : la baseline du notebook 47 ne lisait donc que le début des documents longs. Pour séparer les deux effets du découpage, chaque granularité est évaluée deux fois :

- **tous** : tous les passages, qui couvrent l'intégralité du résumé ;
- **fenêtre** : seulement les passages situés dans la partie que la baseline lisait réellement.

Deux granularités sont testées : 3 phrases (pas de 2) et 2 phrases (pas de 1), avec une phrase de recouvrement.

Le budget est exprimé en **documents** : on descend le classement des passages jusqu'à réunir K documents distincts, comme pour les deux autres pipelines.

**Entrées** (`data/processed/`) : `corpus.json`, `passages_3p.json`, `passages_2p.json`, `jeu_de_test.json`, `courbe_rappel.json`
**Sortie** : `baseline_passages.json`, à replacer dans `data/processed/`
**Matériel** : GPU T4 (Colab gratuit). Aucun modèle de langue, pas d'Ollama.

## Cellule 1 — Installation

In [1]:
!pip install -q sentence-transformers

## Cellule 2 — Vérifier le GPU

In [2]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Modèle de GPU  :", torch.cuda.get_device_name(0))

GPU disponible : True
Modèle de GPU  : Tesla T4


## Cellule 3 — Téléverser les cinq fichiers

In [3]:
from google.colab import files

televerses = files.upload()

ATTENDUS = {"corpus.json", "passages_3p.json", "passages_2p.json",
            "jeu_de_test.json", "courbe_rappel.json"}
manquants = ATTENDUS - set(televerses)
print("Fichiers reçus :", sorted(televerses))
print("Manquants      :", sorted(manquants) if manquants else "aucun")

Saving corpus.json to corpus.json
Saving courbe_rappel.json to courbe_rappel.json
Saving jeu_de_test.json to jeu_de_test.json
Saving passages_2p.json to passages_2p.json
Saving passages_3p.json to passages_3p.json
Fichiers reçus : ['corpus.json', 'courbe_rappel.json', 'jeu_de_test.json', 'passages_2p.json', 'passages_3p.json']
Manquants      : aucun


## Cellule 4 — Charger le modèle

In [4]:
from sentence_transformers import SentenceTransformer

modele = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO", device="cuda")
print("Limite de lecture du modèle :", modele.max_seq_length, "tokens")
print("Tokeniseur rapide           :", modele.tokenizer.is_fast)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Limite de lecture du modèle : 350 tokens
Tokeniseur rapide           : True


## Cellule 5 — Mesurer la troncature

In [5]:
import json
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()   # les textes longs sont attendus ici

LIMITE = modele.max_seq_length

with open("corpus.json", encoding="utf-8") as f:
    corpus = {a["pmid"]: a for a in json.load(f)}

def longueurs(textes):
    """Nombre de tokens de chaque texte, jetons spéciaux compris, sans troncature."""
    ids = modele.tokenizer(textes, add_special_tokens=True, truncation=False)["input_ids"]
    return [len(x) for x in ids]

# Les documents entiers, exactement comme le notebook 47 les encodait
pmids = list(corpus)
docs_textes = [corpus[p]["titre"] + "\n\n" + corpus[p]["resume"] for p in pmids]
docs_long = longueurs(docs_textes)
tronques = sum(1 for n in docs_long if n > LIMITE)
print(f"Documents entiers : {len(docs_long)}, "
      f"longueur moyenne {sum(docs_long) / len(docs_long):.0f} tokens")
print(f"  dépassant la limite de {LIMITE} : {tronques} "
      f"({100 * tronques / len(docs_long):.0f} %)")

# Chaque passage : sa propre longueur, et sa position dans le document.
# Il est « dans la fenêtre » si le document, lu jusqu'à la fin de ce
# passage, tient encore dans la limite : la baseline le voyait en entier.
# Le premier passage d'un document y est toujours : la baseline en voyait
# le début, et s'il dépasse la limite, le modèle le tronque exactement là
# où il tronquait le document entier.
PASSAGES = {}
for etiquette in ("3p", "2p"):
    with open(f"passages_{etiquette}.json", encoding="utf-8") as f:
        ps = json.load(f)
    longs = longueurs([p["texte"] for p in ps])
    prefixes = [corpus[p["pmid"]]["titre"] + "\n\n"
                + corpus[p["pmid"]]["resume"][:p["fin_car"]] for p in ps]
    fins = longueurs(prefixes)
    for p, n in zip(ps, fins):
        p["dans_fenetre"] = n <= LIMITE or p["rang"] == 0
    PASSAGES[etiquette] = ps
    dedans = sum(p["dans_fenetre"] for p in ps)
    premiers_longs = sum(1 for p, n in zip(ps, fins) if p["rang"] == 0 and n > LIMITE)
    print(f"\nPassages {etiquette} : {len(ps)}")
    print(f"  dépassant eux-mêmes la limite             : {sum(1 for n in longs if n > LIMITE)}")
    print(f"  situés dans la partie lue par la baseline : {dedans} "
          f"({100 * dedans / len(ps):.0f} %)")
    print(f"  dont premiers passages dépassant la limite : {premiers_longs}")

Documents entiers : 475, longueur moyenne 330 tokens
  dépassant la limite de 350 : 166 (35 %)

Passages 3p : 2248
  dépassant eux-mêmes la limite             : 6
  situés dans la partie lue par la baseline : 1783 (79 %)
  dont premiers passages dépassant la limite : 3

Passages 2p : 4264
  dépassant eux-mêmes la limite             : 5
  situés dans la partie lue par la baseline : 3493 (82 %)
  dont premiers passages dépassant la limite : 0


## Cellule 6 — Encoder et classer

In [6]:
import torch

with open("jeu_de_test.json", encoding="utf-8") as f:
    questions = json.load(f)

BUDGETS = (1, 3, 5, 10, 20)
KMAX = max(BUDGETS)

vq = modele.encode([q["question"] for q in questions],
                   convert_to_tensor=True, normalize_embeddings=True)

def encoder(textes, taille_lot):
    return modele.encode(textes, batch_size=taille_lot, convert_to_tensor=True,
                         normalize_embeddings=True, show_progress_bar=True)

def documents_par_budget(ordre, pmid_de, K):
    """Descend le classement des unités jusqu'à réunir K documents distincts."""
    vus, deja = [], set()
    for i in ordre:
        pm = pmid_de[i]
        if pm not in deja:
            deja.add(pm)
            vus.append(pm)
            if len(vus) == K:
                break
    return vus

def classer(scores, pmid_de):
    ordres = torch.argsort(scores, dim=1, descending=True).cpu().tolist()
    finis = torch.isfinite(scores).cpu().tolist()
    return {q["id"]: documents_par_budget([i for i in o if f[i]], pmid_de, KMAX)
            for q, o, f in zip(questions, ordres, finis)}

CLASSEMENTS = {}

# Contrôle : la baseline d'origine, recalculée à l'identique du notebook 47
vd = encoder(docs_textes, 32)
CLASSEMENTS["resume_entier"] = classer(vq @ vd.T, pmids)

for etiquette, ps in PASSAGES.items():
    vp = encoder([p["texte"] for p in ps], 64)
    scores = vq @ vp.T                                  # produit scalaire = cosinus
    pmid_de = [p["pmid"] for p in ps]
    CLASSEMENTS[f"passages_{etiquette}"] = classer(scores, pmid_de)
    hors = torch.tensor([not p["dans_fenetre"] for p in ps], device=scores.device)
    CLASSEMENTS[f"passages_{etiquette}_fenetre"] = classer(
        scores.masked_fill(hors, float("-inf")), pmid_de)

print("\nClassements calculés :", list(CLASSEMENTS))

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Batches:   0%|          | 0/67 [00:00<?, ?it/s]


Classements calculés : ['resume_entier', 'passages_3p', 'passages_3p_fenetre', 'passages_2p', 'passages_2p_fenetre']


## Cellule 7 — Résultats

In [8]:
def rappel(classement, K, categorie=None):
    qs = [q for q in questions if categorie is None or q["type"] == categorie]
    return sum(len(set(q["pmids"]) & set(classement[q["id"]][:K])) / len(q["pmids"])
               for q in qs) / len(qs)

with open("courbe_rappel.json", encoding="utf-8") as f:
    courbe = json.load(f)
PUBLIE = {"vect": {l["id"]: l["vect"] for l in courbe["detail"]},
          "graphe": {l["id"]: l["graphe"] for l in courbe["detail"]}}

print("CONTRÔLE — baseline d'origine recalculée ici, contre les valeurs publiées")
identique = True
for K in BUDGETS:
    r_calc, r_pub = rappel(CLASSEMENTS["resume_entier"], K), rappel(PUBLIE["vect"], K)
    identique &= abs(r_calc - r_pub) < 1e-9
    print(f"  K={K:>2}   recalculé {r_calc:.4f}   publié {r_pub:.4f}")
print("  ->", "IDENTIQUE" if identique else "DIFFÉRENT : me le signaler avant toute interprétation")

COLONNES = [("résumé entier", CLASSEMENTS["resume_entier"]),
            ("3p fenêtre",    CLASSEMENTS["passages_3p_fenetre"]),
            ("3p tous",       CLASSEMENTS["passages_3p"]),
            ("2p fenêtre",    CLASSEMENTS["passages_2p_fenetre"]),
            ("2p tous",       CLASSEMENTS["passages_2p"]),
            ("graphe",        PUBLIE["graphe"])]

print("\nRAPPEL SELON LE BUDGET DOCUMENTAIRE")
print(f"{'K':>13}" + "".join(f"{nom:>15}" for nom, _ in COLONNES))
for K in BUDGETS:
    print(f"{K:>13}" + "".join(f"{rappel(c, K):>15.2f}" for _, c in COLONNES))

print("\nRAPPEL@5 PAR CATÉGORIE")
for cat in ("single-hop", "pont", "intersection"):
    print(f"{cat:>13}" + "".join(f"{rappel(c, 5, cat):>15.2f}" for _, c in COLONNES))

print("\nDÉCOMPOSITION DE L'ÉCART AU RANG 5")
base = rappel(CLASSEMENTS["resume_entier"], 5)
graphe = rappel(PUBLIE["graphe"], 5)
for e in ("3p", "2p"):
    fen = rappel(CLASSEMENTS[f"passages_{e}_fenetre"], 5)
    tous = rappel(CLASSEMENTS[f"passages_{e}"], 5)
    print(f"  {e} : découpage de la partie lue {fen - base:+.2f} | "
          f"ajout de la partie non lue {tous - fen:+.2f} | "
          f"écart restant au graphe {graphe - tous:+.2f}")

CONTRÔLE — baseline d'origine recalculée ici, contre les valeurs publiées
  K= 1   recalculé 0.3750   publié 0.3750
  K= 3   recalculé 0.5250   publié 0.5250
  K= 5   recalculé 0.5500   publié 0.5500
  K=10   recalculé 0.6000   publié 0.6000
  K=20   recalculé 0.7667   publié 0.7667
  -> IDENTIQUE

RAPPEL SELON LE BUDGET DOCUMENTAIRE
            K  résumé entier     3p fenêtre        3p tous     2p fenêtre        2p tous         graphe
            1           0.38           0.38           0.33           0.45           0.42           0.45
            3           0.53           0.57           0.57           0.57           0.57           0.78
            5           0.55           0.60           0.60           0.62           0.62           0.83
           10           0.60           0.72           0.72           0.75           0.72           0.88
           20           0.77           0.92           0.89           0.87           0.87           0.95

RAPPEL@5 PAR CATÉGORIE
   single-hop   

## Cellule 8 — Enregistrer et télécharger

In [9]:
resultat = {
    "modele": "pritamdeka/S-PubMedBert-MS-MARCO",
    "limite_tokens": LIMITE,
    "documents_depassant_la_limite": tronques,
    "controle_baseline_identique": identique,
    "classements": CLASSEMENTS,
    "dans_fenetre": {e: {p["id"]: p["dans_fenetre"] for p in ps}
                     for e, ps in PASSAGES.items()},
}
with open("baseline_passages.json", "w", encoding="utf-8") as f:
    json.dump(resultat, f, ensure_ascii=False, indent=1)
files.download("baseline_passages.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>